In [ ]:
# -*- coding: utf-8 -*-
r"""
NPC重度急性口腔黏膜炎3D dose-map项目
497例历史影像候选池统一预处理与NPZ重新生成
==========================================================

最终固定预处理方案
------------------
目标spacing（X,Y,Z）：2.0 × 2.0 × 3.0 mm
patch大小（X,Y,Z）：80 × 112 × 64 voxels
numpy/PyTorch空间顺序：Z,Y,X = 64 × 112 × 80
裁剪中心：oral cavity三维bounding box物理中心

NPZ影像通道
-----------
ct   : float32，HU截断[-1000, 2000]后线性映射到[-1, 1]
dose : float32，统一转换为Gy后除以70，截断到[0, 1.2]
oral : uint8，0/1
gtv  : uint8，0/1

Historical preprocessing pool
-----------------------------
Imaging preprocessing was completed for 497 imaging-eligible candidate cases
before final analytic cohort definition. Cohort and outcome fields stored at
this stage are legacy metadata and are not used for final model development or
evaluation. Final analytic cohort membership and labels are determined
exclusively from the frozen final master in downstream scripts.

The historical preprocessing source encoded A+B=269 as "Development" and
C=228 as "External". These fields are retained only for audit compatibility
and are deliberately ignored downstream.

A configured preprocessing-only exclusion ID is checked against the historical
source-QC records. It is not an exclusion category in the final 540-to-489
analytic cohort flow.

运行依赖
--------
pip install SimpleITK pandas numpy openpyxl matplotlib tqdm
"""

from pathlib import Path
import os
from datetime import datetime
import gc
import hashlib
import json
import re
import shutil
import traceback
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk

from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter


warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. 项目路径与输出路径
# ============================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

PROJECT_DIR = require_env_path("NPC_PROJECT_ROOT")

# ------------------------------------------------------------
# 历史预处理冻结master：固定文件名，用于重建497例影像候选池
# ------------------------------------------------------------

MASTER_XLSX = (
    PROJECT_DIR
    / "NPC_3DCNN_mucositis_master_frozen.xlsx"
)

# 历史预处理Frozen Master的SHA256。
# 文件内容发生任何变化时，程序会停止，防止误用旧版或错误版master。
EXPECTED_MASTER_SHA256 = (
    "366fdeb4b230191575b3f0f0d255b698"
    "610a49ec6d56cc78b9c1b0c8ca89536f"
)

STRICT_MASTER_SHA256 = True

# ------------------------------------------------------------
# 历史预处理候选池输出目录
# ------------------------------------------------------------

OUTPUT_VERSION = "v2"

OUTPUT_ROOT = (
    PROJECT_DIR
    / f"preprocessed_497_2x2x3_patch80x112x64_{OUTPUT_VERSION}"
)

NPZ_DIR = (
    OUTPUT_ROOT
    / "npz"
)

OVERLAY_DIR = (
    OUTPUT_ROOT
    / "NPZ_QC_overlays"
)

REPORT_XLSX = (
    OUTPUT_ROOT
    / "preprocessing_manifest_497.xlsx"
)

INDEX_CSV = (
    OUTPUT_ROOT
    / "dataset_index_497.csv"
)

AUDIT_CSV = (
    OUTPUT_ROOT
    / "npz_audit_497.csv"
)

CONFIG_JSON = (
    OUTPUT_ROOT
    / "preprocessing_config_497.json"
)

RUN_LOG = (
    OUTPUT_ROOT
    / "preprocessing_run_log_497.txt"
)


# ============================================================
# 2. 输出保护
# ============================================================

# False：如果输出目录已经存在，直接停止，防止覆盖。
# 正式首次运行保持False。
ALLOW_OVERWRITE_OUTPUT = False

# True：为全部497例保存一张三平面组合QC图。
SAVE_ALL_OVERLAYS = True


# ============================================================
# 3. 固定预处理参数
# ============================================================

TARGET_SPACING_XYZ = (
    2.0,
    2.0,
    3.0,
)

PATCH_SIZE_XYZ = (
    80,
    112,
    64,
)

EXPECTED_ARRAY_SHAPE_ZYX = (
    PATCH_SIZE_XYZ[2],
    PATCH_SIZE_XYZ[1],
    PATCH_SIZE_XYZ[0],
)

CT_CLIP_MIN_HU = -1000.0
CT_CLIP_MAX_HU = 2000.0

DOSE_NORMALIZATION_GY = 70.0
DOSE_NORMALIZED_MAX = 1.2

# oral必须基本完全保留
ORAL_RETENTION_REQUIRED = 0.999

# GTV低于99%时记为warning，但不自动剔除
GTV_RETENTION_WARNING = 0.990


# ============================================================
# 4. 几何比较容差
# ============================================================

SPACING_ATOL_MM = 1e-5
ORIGIN_ATOL_MM = 1e-3
DIRECTION_ATOL = 1e-5


# ============================================================
# 5. 历史预处理候选池严格预期（不是最终489例分析队列）
# ============================================================

PREPROCESSING_EXCLUDED_ID = int(os.environ["NPC_PREPROCESSING_EXCLUDED_ID"])

EXPECTED_MASTER_TOTAL = 540
EXPECTED_EXCLUDED_TOTAL = 43
EXPECTED_INCLUDED_TOTAL = 497

EXPECTED_COHORT_COUNTS = {
    "Development": 269,
    "External": 228,
}

EXPECTED_MODEL_CENTER_COUNTS = {
    "A": 188,
    "B": 81,
    "C": 228,
}

EXPECTED_MODEL_CENTER_LABEL_COUNTS = {
    ("A", 0): 95,
    ("A", 1): 93,

    ("B", 0): 38,
    ("B", 1): 43,

    ("C", 0): 115,
    ("C", 1): 113,
}

EXPECTED_TOTAL_LABEL_COUNTS = {
    0: 248,
    1: 249,
}

# Excluding the configured historical preprocessing-only case，原影像QC状态应为：
# 原498例：496 PASS + 2 WARN
# The configured preprocessing-only excluded case originally had PASS status
# 因此历史预处理候选池497例：495 PASS + 2 WARN
EXPECTED_POOL_QC_STATUS_COUNTS = {
    "PASS": 495,
    "WARN": 2,
    "FAIL": 0,
}

EXPECTED_SOURCE_QC_WARNING_IDS = {
    int(value.strip())
    for value in os.environ["NPC_SOURCE_QC_WARNING_IDS"].split(",")
    if value.strip()
}

# Expected warning cases are configured outside the public source.
EXPECTED_PREPROCESSING_WARNING_IDS = {
    int(value.strip())
    for value in os.environ["NPC_PREPROCESSING_WARNING_IDS"].split(",")
    if value.strip()
}


# ============================================================
# 6. 通用辅助函数
# ============================================================

def sha256_file(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """计算文件SHA256。"""
    sha = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            block = file.read(chunk_size)

            if not block:
                break

            sha.update(block)

    return sha.hexdigest()


def is_blank(value) -> bool:
    """判断Excel单元格是否为空。"""
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except Exception:
        pass

    return str(value).strip() == ""


def normalize_patient_id(value) -> str:
    """统一patient_id格式。"""
    if is_blank(value):
        return ""

    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    try:
        number = float(text)

        if number.is_integer():
            return str(int(number))

    except Exception:
        pass

    return text


def normalize_header(value) -> str:
    """用于兼容字段名大小写、空格、下划线。"""
    if value is None:
        return ""

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def find_column(
    dataframe: pd.DataFrame,
    candidates,
    required=False,
):
    """从候选字段名中寻找实际字段。"""
    normalized_columns = {
        normalize_header(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        normalized_candidate = normalize_header(
            candidate
        )

        if normalized_candidate in normalized_columns:
            return normalized_columns[
                normalized_candidate
            ]

    if required:
        raise KeyError(
            "没有找到必要字段。\n"
            f"候选字段：{candidates}\n"
            f"当前字段：{list(dataframe.columns)}"
        )

    return None


def safe_float(
    value,
    default=np.nan,
):
    """安全转换为float。"""
    try:
        number = float(value)

        if np.isfinite(number):
            return number

    except Exception:
        pass

    return default


def safe_int(
    value,
    default=None,
):
    """安全转换为int。"""
    try:
        return int(float(value))

    except Exception:
        return default


def require_float(
    value,
    field_name,
    patient_id,
):
    """必要数值字段必须存在。"""
    number = safe_float(value)

    if not np.isfinite(number):
        raise ValueError(
            f"病例{patient_id}字段"
            f"{field_name}不是有效数值：{value}"
        )

    return float(number)


def require_int(
    value,
    field_name,
    patient_id,
):
    """必要整数字段必须存在。"""
    number = safe_int(value)

    if number is None:
        raise ValueError(
            f"病例{patient_id}字段"
            f"{field_name}不是有效整数：{value}"
        )

    return int(number)


def normalize_text(value) -> str:
    """统一普通文本。"""
    if is_blank(value):
        return ""

    return str(value).strip()


def read_3d_image(
    path: Path,
) -> sitk.Image:
    """读取三维医学影像。"""
    if path is None:
        raise FileNotFoundError(
            "影像路径为空"
        )

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"找不到文件：{path}"
        )

    if not path.is_file():
        raise FileNotFoundError(
            f"路径不是文件：{path}"
        )

    image = sitk.ReadImage(
        str(path)
    )

    if image.GetDimension() != 3:
        raise ValueError(
            f"不是三维影像：{path}；"
            f"dimension={image.GetDimension()}"
        )

    return image


def same_geometry(
    reference: sitk.Image,
    image: sitk.Image,
) -> bool:
    """比较size、spacing、origin、direction。"""
    return (
        tuple(reference.GetSize())
        == tuple(image.GetSize())

        and np.allclose(
            reference.GetSpacing(),
            image.GetSpacing(),
            atol=SPACING_ATOL_MM,
            rtol=0,
        )

        and np.allclose(
            reference.GetOrigin(),
            image.GetOrigin(),
            atol=ORIGIN_ATOL_MM,
            rtol=0,
        )

        and np.allclose(
            reference.GetDirection(),
            image.GetDirection(),
            atol=DIRECTION_ATOL,
            rtol=0,
        )
    )


def geometry_detail(
    reference: sitk.Image,
    image: sitk.Image,
):
    """记录两幅影像几何差异。"""
    return {
        "size_equal": (
            tuple(reference.GetSize())
            == tuple(image.GetSize())
        ),

        "max_spacing_diff_mm": float(
            np.max(
                np.abs(
                    np.asarray(
                        reference.GetSpacing(),
                        dtype=float,
                    )
                    -
                    np.asarray(
                        image.GetSpacing(),
                        dtype=float,
                    )
                )
            )
        ),

        "max_origin_diff_mm": float(
            np.max(
                np.abs(
                    np.asarray(
                        reference.GetOrigin(),
                        dtype=float,
                    )
                    -
                    np.asarray(
                        image.GetOrigin(),
                        dtype=float,
                    )
                )
            )
        ),

        "max_direction_diff": float(
            np.max(
                np.abs(
                    np.asarray(
                        reference.GetDirection(),
                        dtype=float,
                    )
                    -
                    np.asarray(
                        image.GetDirection(),
                        dtype=float,
                    )
                )
            )
        ),
    }


def binary_array(
    image: sitk.Image,
) -> np.ndarray:
    """mask转换为0/1数组，顺序Z,Y,X。"""
    array = sitk.GetArrayFromImage(
        image
    ).astype(np.float32)

    return (
        array > 0.5
    ).astype(np.uint8)


def mask_bbox_center_physical(
    mask_image: sitk.Image,
):
    """
    根据mask三维包围盒计算中心。

    返回：
    1. center_physical_xyz
    2. center_continuous_index_xyz
    """
    array = binary_array(
        mask_image
    )

    coordinates_zyx = np.argwhere(
        array > 0
    )

    if coordinates_zyx.size == 0:
        raise ValueError(
            "oral mask为空，无法确定裁剪中心"
        )

    minimum_zyx = (
        coordinates_zyx
        .min(axis=0)
        .astype(float)
    )

    maximum_zyx = (
        coordinates_zyx
        .max(axis=0)
        .astype(float)
    )

    center_zyx = (
        minimum_zyx
        +
        maximum_zyx
    ) / 2.0

    center_xyz = (
        float(center_zyx[2]),
        float(center_zyx[1]),
        float(center_zyx[0]),
    )

    center_physical_xyz = (
        mask_image
        .TransformContinuousIndexToPhysicalPoint(
            center_xyz
        )
    )

    return (
        tuple(
            float(value)
            for value
            in center_physical_xyz
        ),

        tuple(
            float(value)
            for value
            in center_xyz
        ),
    )


def create_patch_reference(
    ct_image: sitk.Image,
    center_physical_xyz,
) -> sitk.Image:
    """
    创建固定patch参考网格。

    patch方向继承CT；
    patch中心与oral包围盒物理中心一致。
    """
    direction_matrix = np.asarray(
        ct_image.GetDirection(),
        dtype=float,
    ).reshape(3, 3)

    spacing = np.asarray(
        TARGET_SPACING_XYZ,
        dtype=float,
    )

    size = np.asarray(
        PATCH_SIZE_XYZ,
        dtype=float,
    )

    center_index = (
        size - 1.0
    ) / 2.0

    center_physical = np.asarray(
        center_physical_xyz,
        dtype=float,
    )

    origin = (
        center_physical
        -
        direction_matrix
        @ (
            center_index
            * spacing
        )
    )

    reference = sitk.Image(
        [
            int(value)
            for value
            in PATCH_SIZE_XYZ
        ],
        sitk.sitkFloat32,
    )

    reference.SetSpacing(
        tuple(
            float(value)
            for value
            in TARGET_SPACING_XYZ
        )
    )

    reference.SetDirection(
        ct_image.GetDirection()
    )

    reference.SetOrigin(
        tuple(
            float(value)
            for value
            in origin
        )
    )

    return reference


def resample_to_reference(
    image: sitk.Image,
    reference: sitk.Image,
    interpolator,
    default_value,
    output_pixel_type,
) -> sitk.Image:
    """重采样到固定patch空间。"""
    return sitk.Resample(
        image,
        reference,
        sitk.Transform(),
        interpolator,
        float(default_value),
        output_pixel_type,
    )


def patch_retained_fraction(
    original_mask: sitk.Image,
    patch_reference: sitk.Image,
) -> float:
    """
    计算原始ROI落入最终patch的比例。

    将patch视野反向映射到原始mask空间，
    避免因目标spacing改变造成体素比例误差。
    """
    patch_ones_array = np.ones(
        EXPECTED_ARRAY_SHAPE_ZYX,
        dtype=np.uint8,
    )

    patch_ones_image = sitk.GetImageFromArray(
        patch_ones_array
    )

    patch_ones_image.CopyInformation(
        patch_reference
    )

    patch_fov_on_original = sitk.Resample(
        patch_ones_image,
        original_mask,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        sitk.sitkUInt8,
    )

    original_array = (
        binary_array(original_mask) > 0
    )

    fov_array = (
        sitk.GetArrayFromImage(
            patch_fov_on_original
        ) > 0
    )

    denominator = int(
        original_array.sum()
    )

    if denominator == 0:
        return np.nan

    numerator = int(
        np.logical_and(
            original_array,
            fov_array,
        ).sum()
    )

    return float(
        numerator / denominator
    )


def physical_volume_cc(
    mask_image: sitk.Image,
) -> float:
    """计算mask物理体积，单位cc。"""
    array = binary_array(
        mask_image
    )

    voxel_volume_mm3 = float(
        np.prod(
            mask_image.GetSpacing()
        )
    )

    return float(
        array.sum()
        * voxel_volume_mm3
        / 1000.0
    )


def infer_dose_scale_to_gy(
    dose_image: sitk.Image,
    qc_raw_to_cgy_scale,
):
    """
    确定原始dose转换至Gy的比例。

    QC字段dose_raw_to_cgy_scale：
    - 原始dose为cGy时：值通常为1
    - 原始dose为Gy时：值通常为100

    因此：
    dose_scale_to_Gy
    = dose_raw_to_cgy_scale / 100
    """
    qc_scale = safe_float(
        qc_raw_to_cgy_scale
    )

    if (
        np.isfinite(qc_scale)
        and qc_scale > 0
    ):
        scale_to_gy = (
            qc_scale / 100.0
        )

        if (
            scale_to_gy < 0.0001
            or scale_to_gy > 10
        ):
            raise ValueError(
                "QC给出的dose转换比例异常："
                f"raw_to_cGy={qc_scale}"
            )

        return (
            float(scale_to_gy),
            "final_QC_report",
        )

    # QC字段缺失时的后备判断
    array = sitk.GetArrayFromImage(
        dose_image
    ).astype(np.float32)

    finite = array[
        np.isfinite(array)
    ]

    if finite.size == 0:
        raise ValueError(
            "dose中没有有限数值"
        )

    percentile_999 = float(
        np.percentile(
            finite,
            99.9,
        )
    )

    if percentile_999 > 200:
        return (
            0.01,
            "inferred_cGy",
        )

    return (
        1.0,
        "inferred_Gy",
    )


def prepare_arrays(
    ct_patch: sitk.Image,
    dose_patch: sitk.Image,
    oral_patch: sitk.Image,
    gtv_patch: sitk.Image,
    dose_scale_to_gy: float,
):
    """生成最终NPZ中的四个影像数组。"""
    ct_raw = (
        sitk.GetArrayFromImage(
            ct_patch
        )
        .astype(np.float32)
    )

    dose_raw = (
        sitk.GetArrayFromImage(
            dose_patch
        )
        .astype(np.float32)
    )

    oral = binary_array(
        oral_patch
    )

    gtv = binary_array(
        gtv_patch
    )

    ct_raw = np.nan_to_num(
        ct_raw,
        nan=CT_CLIP_MIN_HU,
        posinf=CT_CLIP_MAX_HU,
        neginf=CT_CLIP_MIN_HU,
    )

    dose_raw = np.nan_to_num(
        dose_raw,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # --------------------------------------------------------
    # CT：[-1000,2000]映射到[-1,1]
    # --------------------------------------------------------

    ct_clipped = np.clip(
        ct_raw,
        CT_CLIP_MIN_HU,
        CT_CLIP_MAX_HU,
    )

    ct_normalized = (
        2.0
        * (
            ct_clipped
            - CT_CLIP_MIN_HU
        )
        / (
            CT_CLIP_MAX_HU
            - CT_CLIP_MIN_HU
        )
        - 1.0
    ).astype(np.float32)

    # --------------------------------------------------------
    # Dose：原单位 -> Gy -> /70 -> [0,1.2]
    # --------------------------------------------------------

    dose_gy = (
        dose_raw
        * float(
            dose_scale_to_gy
        )
    )

    dose_gy = np.clip(
        dose_gy,
        0.0,
        (
            DOSE_NORMALIZATION_GY
            * DOSE_NORMALIZED_MAX
        ),
    )

    dose_normalized = np.clip(
        dose_gy
        / DOSE_NORMALIZATION_GY,
        0.0,
        DOSE_NORMALIZED_MAX,
    ).astype(np.float32)

    arrays = {
        "ct": (
            ct_normalized
            .astype(np.float32)
        ),

        "dose": (
            dose_normalized
            .astype(np.float32)
        ),

        "oral": (
            oral.astype(np.uint8)
        ),

        "gtv": (
            gtv.astype(np.uint8)
        ),

        # 以下两个数组只用于报告和绘图，
        # 不作为正式训练通道另存
        "ct_hu_for_qc": (
            ct_clipped
            .astype(np.float32)
        ),

        "dose_gy_for_qc": (
            dose_gy
            .astype(np.float32)
        ),
    }

    for name in [
        "ct",
        "dose",
        "oral",
        "gtv",
    ]:
        if (
            arrays[name].shape
            != EXPECTED_ARRAY_SHAPE_ZYX
        ):
            raise RuntimeError(
                f"{name}输出形状错误："
                f"{arrays[name].shape}；"
                f"预期="
                f"{EXPECTED_ARRAY_SHAPE_ZYX}"
            )

    if not np.isfinite(
        arrays["ct"]
    ).all():
        raise RuntimeError(
            "CT归一化后存在NaN或Inf"
        )

    if not np.isfinite(
        arrays["dose"]
    ).all():
        raise RuntimeError(
            "dose归一化后存在NaN或Inf"
        )

    if int(
        arrays["oral"].sum()
    ) <= 0:
        raise RuntimeError(
            "输出oral mask为空"
        )

    if int(
        arrays["gtv"].sum()
    ) <= 0:
        raise RuntimeError(
            "输出GTV mask为空"
        )

    if (
        float(
            arrays["ct"].min()
        ) < -1.0001
        or
        float(
            arrays["ct"].max()
        ) > 1.0001
    ):
        raise RuntimeError(
            "CT归一化范围异常"
        )

    if (
        float(
            arrays["dose"].min()
        ) < -1e-6
        or
        float(
            arrays["dose"].max()
        )
        > DOSE_NORMALIZED_MAX + 1e-6
    ):
        raise RuntimeError(
            "dose归一化范围异常"
        )

    return arrays


def scalar_text(value) -> str:
    """读取NPZ中的字符串标量。"""
    array = np.asarray(
        value
    )

    if array.size != 1:
        return str(array)

    return str(
        array.reshape(-1)[0]
    )


def save_npz_atomic(
    output_path: Path,
    arrays,
    metadata,
):
    """
    原子方式保存NPZ。

    先写临时文件，写完后再改名，
    避免程序中断留下损坏文件。
    """
    temporary_path = (
        output_path.parent
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "wb",
    ) as file:

        np.savez_compressed(
            file,

            # ------------------------------------------------
            # 正式影像通道
            # ------------------------------------------------

            ct=arrays[
                "ct"
            ].astype(
                np.float32
            ),

            dose=arrays[
                "dose"
            ].astype(
                np.float32
            ),

            oral=arrays[
                "oral"
            ].astype(
                np.uint8
            ),

            gtv=arrays[
                "gtv"
            ].astype(
                np.uint8
            ),

            # ------------------------------------------------
            # 病例标识及标签
            # ------------------------------------------------

            patient_id=np.asarray(
                metadata[
                    "patient_id"
                ],
                dtype=np.int32,
            ),

            center=np.asarray(
                metadata[
                    "original_center"
                ],
                dtype=np.int8,
            ),

            original_center=np.asarray(
                metadata[
                    "original_center"
                ],
                dtype=np.int8,
            ),

            model_center=np.asarray(
                metadata[
                    "model_center"
                ]
            ),

            cohort=np.asarray(
                metadata[
                    "cohort"
                ]
            ),

            label=np.asarray(
                metadata[
                    "label"
                ],
                dtype=np.int8,
            ),

            severe_mucositis=np.asarray(
                metadata[
                    "label"
                ],
                dtype=np.int8,
            ),

            # ------------------------------------------------
            # 治疗参数
            # ------------------------------------------------

            prescription_dose_cgy=np.asarray(
                metadata[
                    "prescription_dose_cgy"
                ],
                dtype=np.float32,
            ),

            fraction_dose_cgy=np.asarray(
                metadata[
                    "fraction_dose_cgy"
                ],
                dtype=np.float32,
            ),

            fractions=np.asarray(
                metadata[
                    "fractions"
                ],
                dtype=np.int16,
            ),

            rt_technique=np.asarray(
                metadata[
                    "rt_technique"
                ]
            ),

            rt_start_year=np.asarray(
                metadata[
                    "rt_start_year"
                ],
                dtype=np.int16,
            ),

            # ------------------------------------------------
            # 来源QC状态
            # ------------------------------------------------

            source_qc_status=np.asarray(
                metadata[
                    "source_qc_status"
                ]
            ),

            source_qc_warning=np.asarray(
                metadata[
                    "source_qc_warning"
                ]
            ),

            # ------------------------------------------------
            # 空间信息
            # ------------------------------------------------

            target_spacing_xyz=np.asarray(
                TARGET_SPACING_XYZ,
                dtype=np.float32,
            ),

            patch_size_xyz=np.asarray(
                PATCH_SIZE_XYZ,
                dtype=np.int16,
            ),

            patch_origin_xyz=np.asarray(
                metadata[
                    "patch_origin_xyz"
                ],
                dtype=np.float64,
            ),

            patch_direction=np.asarray(
                metadata[
                    "patch_direction"
                ],
                dtype=np.float64,
            ),

            oral_center_physical_xyz=np.asarray(
                metadata[
                    "oral_center_physical_xyz"
                ],
                dtype=np.float64,
            ),

            oral_center_index_xyz=np.asarray(
                metadata[
                    "oral_center_index_xyz"
                ],
                dtype=np.float64,
            ),

            # ------------------------------------------------
            # 归一化参数
            # ------------------------------------------------

            ct_clip_hu=np.asarray(
                [
                    CT_CLIP_MIN_HU,
                    CT_CLIP_MAX_HU,
                ],
                dtype=np.float32,
            ),

            dose_normalization_gy=np.asarray(
                DOSE_NORMALIZATION_GY,
                dtype=np.float32,
            ),

            dose_normalized_max=np.asarray(
                DOSE_NORMALIZED_MAX,
                dtype=np.float32,
            ),

            dose_scale_to_gy=np.asarray(
                metadata[
                    "dose_scale_to_gy"
                ],
                dtype=np.float32,
            ),
        )

    temporary_path.replace(
        output_path
    )


def largest_area_index(
    mask: np.ndarray,
    plane: str,
) -> int:
    """寻找oral最大截面积层。"""
    if plane == "axial":
        return int(
            np.argmax(
                mask.sum(
                    axis=(1, 2)
                )
            )
        )

    if plane == "coronal":
        return int(
            np.argmax(
                mask.sum(
                    axis=(0, 2)
                )
            )
        )

    if plane == "sagittal":
        return int(
            np.argmax(
                mask.sum(
                    axis=(0, 1)
                )
            )
        )

    raise ValueError(
        f"未知平面：{plane}"
    )


def extract_plane(
    array_zyx: np.ndarray,
    plane: str,
    index: int,
):
    """提取指定平面。"""
    if plane == "axial":
        return array_zyx[
            index,
            :,
            :,
        ]

    if plane == "coronal":
        return array_zyx[
            :,
            index,
            :,
        ]

    if plane == "sagittal":
        return array_zyx[
            :,
            :,
            index,
        ]

    raise ValueError(
        f"未知平面：{plane}"
    )


def save_combined_overlay(
    arrays,
    patient_id: int,
    output_path: Path,
    status_text: str,
):
    """
    保存轴位、冠状位和矢状位组合图。

    oral：实线轮廓
    GTV：虚线轮廓
    """
    planes = [
        "axial",
        "coronal",
        "sagittal",
    ]

    figure, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6),
    )

    for axis, plane in zip(
        axes,
        planes,
    ):
        index = largest_area_index(
            arrays["oral"],
            plane,
        )

        ct_2d = extract_plane(
            arrays[
                "ct_hu_for_qc"
            ],
            plane,
            index,
        )

        dose_2d = extract_plane(
            arrays[
                "dose_gy_for_qc"
            ],
            plane,
            index,
        )

        oral_2d = extract_plane(
            arrays["oral"],
            plane,
            index,
        )

        gtv_2d = extract_plane(
            arrays["gtv"],
            plane,
            index,
        )

        axis.imshow(
            ct_2d,
            cmap="gray",
            vmin=-1000,
            vmax=1000,
            origin="lower",
        )

        dose_masked = np.ma.masked_where(
            dose_2d <= 1.0,
            dose_2d,
        )

        axis.imshow(
            dose_masked,
            alpha=0.25,
            cmap="turbo",
            vmin=0,
            vmax=75,
            origin="lower",
        )

        if np.any(oral_2d):
            axis.contour(
                oral_2d.astype(float),
                levels=[0.5],
                linewidths=1.2,
                linestyles="-",
                origin="lower",
            )

        if np.any(gtv_2d):
            axis.contour(
                gtv_2d.astype(float),
                levels=[0.5],
                linewidths=1.2,
                linestyles="--",
                origin="lower",
            )

        axis.set_title(
            f"{plane.capitalize()} | "
            f"index={index}"
        )

        axis.axis("off")

    figure.suptitle(
        f"Patient {patient_id} | "
        f"{status_text}\n"
        "CT + dose + oral(solid) + GTV(dashed)",
        fontsize=14,
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=140,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


def format_excel_report(
    excel_path: Path,
):
    """格式化最终Excel报告。"""
    workbook = load_workbook(
        excel_path
    )

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    success_fill = PatternFill(
        fill_type="solid",
        fgColor="E2F0D9",
    )

    warning_fill = PatternFill(
        fill_type="solid",
        fgColor="FFF2CC",
    )

    failure_fill = PatternFill(
        fill_type="solid",
        fgColor="F4CCCC",
    )

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"

        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for cell in worksheet[1]:
            cell.fill = header_fill

            cell.font = Font(
                bold=True
            )

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
            )

        for column_cells in worksheet.columns:
            maximum_length = 0

            for cell in list(
                column_cells
            )[:1000]:
                if cell.value is not None:
                    maximum_length = max(
                        maximum_length,
                        len(str(cell.value)),
                    )

            width = min(
                max(
                    maximum_length + 2,
                    10,
                ),
                45,
            )

            worksheet.column_dimensions[
                get_column_letter(
                    column_cells[0].column
                )
            ].width = width

        if worksheet.title in {
            "Manifest",
            "Audit",
        }:
            status_column = None

            for cell in worksheet[1]:
                if cell.value in {
                    "status",
                    "audit_status",
                }:
                    status_column = (
                        cell.column
                    )

                    break

            if status_column is not None:
                for row_number in range(
                    2,
                    worksheet.max_row + 1,
                ):
                    status_cell = (
                        worksheet.cell(
                            row=row_number,
                            column=status_column,
                        )
                    )

                    value = str(
                        status_cell.value
                    )

                    if value in {
                        "success",
                        "pass",
                    }:
                        status_cell.fill = (
                            success_fill
                        )

                    elif (
                        "warning" in value
                        or value == "WARN"
                    ):
                        status_cell.fill = (
                            warning_fill
                        )

                    elif value in {
                        "failed",
                        "FAIL",
                    }:
                        status_cell.fill = (
                            failure_fill
                        )

    workbook.save(
        excel_path
    )


# ============================================================
# 7. 锁定并验证最新Frozen Master
# ============================================================

if not MASTER_XLSX.exists():
    raise FileNotFoundError(
        "找不到最新Frozen Master：\n"
        f"{MASTER_XLSX}\n\n"
        "请将当前最终master保存为：\n"
        "NPC_3DCNN_mucositis_master_frozen.xlsx"
    )

master_sha256_precheck = sha256_file(
    MASTER_XLSX
)

if (
    STRICT_MASTER_SHA256
    and
    master_sha256_precheck
    != EXPECTED_MASTER_SHA256
):
    raise RuntimeError(
        "Frozen Master SHA256不匹配，程序停止。\n\n"
        f"文件：{MASTER_XLSX}\n"
        f"当前SHA256：{master_sha256_precheck}\n"
        f"预期SHA256：{EXPECTED_MASTER_SHA256}\n\n"
        "请确认使用的是最终核查标签版master，"
        "不要直接关闭哈希校验。"
    )

print("=" * 88)
print("NPC 497例最新标签版NPZ数据集重新生成")
print("=" * 88)
print("冻结master：", MASTER_XLSX)
print("Master SHA256：", master_sha256_precheck)
print("输出目录：", OUTPUT_ROOT)
print("目标spacing XYZ：", TARGET_SPACING_XYZ)
print("patch大小 XYZ：", PATCH_SIZE_XYZ)
print("输出数组 ZYX：", EXPECTED_ARRAY_SHAPE_ZYX)
print()


# ============================================================
# 8. 正式读取并验证master
# ============================================================

master_df = pd.read_excel(
    MASTER_XLSX,
    dtype=object,
    engine="openpyxl",
)

master_df.columns = [
    str(column).strip()
    for column
    in master_df.columns
]

MASTER_PID_COL = find_column(
    master_df,
    [
        "patient_id",
        "patientid",
        "id",
        "病例编号",
        "编号",
    ],
    required=True,
)

MASTER_ORIGINAL_CENTER_COL = find_column(
    master_df,
    [
        "original_center",
    ],
    required=True,
)

MASTER_MODEL_CENTER_COL = find_column(
    master_df,
    [
        "model_center",
    ],
    required=True,
)

MASTER_COHORT_COL = find_column(
    master_df,
    [
        "cohort",
    ],
    required=True,
)

MASTER_EXCLUDE_COL = find_column(
    master_df,
    [
        "exclude_reason",
        "exclusion_reason",
        "排除原因",
    ],
    required=False,
)

MASTER_LABEL_COL = find_column(
    master_df,
    [
        "severe_mucositis",
    ],
    required=True,
)

MASTER_PRESCRIPTION_COL = find_column(
    master_df,
    [
        "prescription_dose",
    ],
    required=True,
)

MASTER_FRACTION_DOSE_COL = find_column(
    master_df,
    [
        "fraction_dose",
    ],
    required=True,
)

MASTER_FRACTIONS_COL = find_column(
    master_df,
    [
        "fractions",
    ],
    required=True,
)

MASTER_RT_TECHNIQUE_COL = find_column(
    master_df,
    [
        "RT_technique",
        "RT technique",
    ],
    required=True,
)

MASTER_RT_YEAR_COL = find_column(
    master_df,
    [
        "RT_start_year",
        "RT start year",
    ],
    required=True,
)

master_df[
    "patient_id_norm"
] = (
    master_df[
        MASTER_PID_COL
    ]
    .map(
        normalize_patient_id
    )
)

if (
    master_df[
        "patient_id_norm"
    ] == ""
).any():
    raise RuntimeError(
        "master中存在空patient_id"
    )

master_df[
    "patient_id_int"
] = (
    master_df[
        "patient_id_norm"
    ]
    .astype(int)
)

duplicate_master_ids = (
    master_df.loc[
        master_df[
            "patient_id_int"
        ].duplicated(
            keep=False
        ),
        "patient_id_int",
    ]
    .astype(int)
    .tolist()
)

if duplicate_master_ids:
    raise RuntimeError(
        "master存在重复病例编号："
        f"{sorted(set(duplicate_master_ids))}"
    )

if MASTER_EXCLUDE_COL is None:
    included_df = (
        master_df.copy()
    )

    excluded_df = pd.DataFrame(
        columns=master_df.columns
    )

else:
    included_mask = (
        master_df[
            MASTER_EXCLUDE_COL
        ].map(
            is_blank
        )
    )

    included_df = (
        master_df.loc[
            included_mask
        ]
        .copy()
    )

    excluded_df = (
        master_df.loc[
            ~included_mask
        ]
        .copy()
    )

if len(master_df) != EXPECTED_MASTER_TOTAL:
    raise RuntimeError(
        "master总病例数不正确："
        f"实际={len(master_df)}；"
        f"预期={EXPECTED_MASTER_TOTAL}"
    )

if len(excluded_df) != EXPECTED_EXCLUDED_TOTAL:
    raise RuntimeError(
        "master排除病例数不正确："
        f"实际={len(excluded_df)}；"
        f"预期={EXPECTED_EXCLUDED_TOTAL}"
    )

included_df[
    "original_center_int"
] = (
    pd.to_numeric(
        included_df[
            MASTER_ORIGINAL_CENTER_COL
        ],
        errors="raise",
    )
    .astype(int)
)

included_df[
    "model_center_text"
] = (
    included_df[
        MASTER_MODEL_CENTER_COL
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

included_df[
    "cohort_text"
] = (
    included_df[
        MASTER_COHORT_COL
    ]
    .astype(str)
    .str.strip()
)

included_df[
    "cohort_text"
] = (
    included_df[
        "cohort_text"
    ]
    .replace({
        "development":
        "Development",

        "DEVELOPMENT":
        "Development",

        "external":
        "External",

        "EXTERNAL":
        "External",

        "external_validation":
        "External",

        "External validation":
        "External",

        "EXTERNAL_VALIDATION":
        "External",
    })
)

included_df[
    "label_int"
] = (
    pd.to_numeric(
        included_df[
            MASTER_LABEL_COL
        ],
        errors="raise",
    )
    .astype(int)
)

if len(included_df) != EXPECTED_INCLUDED_TOTAL:
    raise RuntimeError(
        "最终纳入病例数不正确："
        f"实际={len(included_df)}；"
        f"预期={EXPECTED_INCLUDED_TOTAL}"
    )

included_ids = set(
    included_df[
        "patient_id_int"
    ].astype(int)
)

if PREPROCESSING_EXCLUDED_ID in included_ids:
    raise RuntimeError(
        f"严重错误：历史预处理阶段配置的排除病例"
        f"{PREPROCESSING_EXCLUDED_ID}"
        "仍在历史预处理候选池中。"
    )

actual_cohort_counts = (
    included_df[
        "cohort_text"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

if (
    actual_cohort_counts
    != EXPECTED_COHORT_COUNTS
):
    raise RuntimeError(
        "cohort数量不正确：\n"
        f"实际={actual_cohort_counts}\n"
        f"预期={EXPECTED_COHORT_COUNTS}"
    )

actual_model_center_counts = (
    included_df[
        "model_center_text"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

if (
    actual_model_center_counts
    != EXPECTED_MODEL_CENTER_COUNTS
):
    raise RuntimeError(
        "模型中心数量不正确：\n"
        f"实际={actual_model_center_counts}\n"
        f"预期={EXPECTED_MODEL_CENTER_COUNTS}"
    )

actual_model_center_label_counts = (
    included_df.groupby(
        [
            "model_center_text",
            "label_int",
        ]
    )
    .size()
    .to_dict()
)

if (
    actual_model_center_label_counts
    != EXPECTED_MODEL_CENTER_LABEL_COUNTS
):
    raise RuntimeError(
        "各模型中心标签数量不正确：\n"
        f"实际="
        f"{actual_model_center_label_counts}\n"
        f"预期="
        f"{EXPECTED_MODEL_CENTER_LABEL_COUNTS}"
    )

actual_total_label_counts = (
    included_df[
        "label_int"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

if (
    actual_total_label_counts
    != EXPECTED_TOTAL_LABEL_COUNTS
):
    raise RuntimeError(
        "总标签数量不正确：\n"
        f"实际="
        f"{actual_total_label_counts}\n"
        f"预期="
        f"{EXPECTED_TOTAL_LABEL_COUNTS}"
    )

print("冻结master验证通过：")
print(f"  master总行数：{len(master_df)}")
print(f"  当前排除：{len(excluded_df)}")
print(f"  最终纳入：{len(included_df)}")
print(f"  Development：{actual_cohort_counts['Development']}")
print(f"  External：{actual_cohort_counts['External']}")
print(f"  Center A：{actual_model_center_counts['A']}")
print(f"  Center B：{actual_model_center_counts['B']}")
print(f"  Center C：{actual_model_center_counts['C']}")
print(f"  Non-severe：{actual_total_label_counts[0]}")
print(f"  Severe：{actual_total_label_counts[1]}")
print("  历史预处理阶段配置的排除病例：未进入497例候选池")
print()


# ============================================================
# 9. 自动寻找适用于历史预处理候选池497例的QC报告
# ============================================================

qc_candidates = [
    path
    for path in PROJECT_DIR.rglob(
        "QC_*_full_report.xlsx"
    )
    if (
        path.is_file()
        and not path.name.startswith("~$")
    )
]

if not qc_candidates:
    raise FileNotFoundError(
        "项目目录中没有找到QC完整报告：\n"
        f"{PROJECT_DIR}"
    )

valid_qc_candidates = []
qc_candidate_messages = []

for qc_candidate in qc_candidates:
    try:
        candidate_qc_df = pd.read_excel(
            qc_candidate,
            sheet_name="Case_level",
            dtype=object,
            engine="openpyxl",
        )

        candidate_qc_df.columns = [
            str(column).strip()
            for column
            in candidate_qc_df.columns
        ]

        candidate_pid_col = find_column(
            candidate_qc_df,
            [
                "patient_id",
            ],
            required=True,
        )

        candidate_status_col = find_column(
            candidate_qc_df,
            [
                "qc_status",
            ],
            required=True,
        )

        candidate_qc_df[
            "_patient_id_norm"
        ] = (
            candidate_qc_df[
                candidate_pid_col
            ]
            .map(
                normalize_patient_id
            )
        )

        candidate_qc_df[
            "_patient_id_int"
        ] = (
            candidate_qc_df[
                "_patient_id_norm"
            ]
            .astype(int)
        )

        candidate_qc_ids = set(
            candidate_qc_df[
                "_patient_id_int"
            ].astype(int)
        )

        missing_final_ids = (
            included_ids
            - candidate_qc_ids
        )

        extra_qc_ids = (
            candidate_qc_ids
            - included_ids
        )

        # 允许旧498例QC多出配置的排除病例；
        # 不允许多出其他病例。
        extra_ids_valid = (
            extra_qc_ids
            .issubset({
                PREPROCESSING_EXCLUDED_ID
            })
        )

        candidate_final_subset = (
            candidate_qc_df.loc[
                candidate_qc_df[
                    "_patient_id_int"
                ].isin(
                    included_ids
                )
            ]
            .copy()
        )

        candidate_final_subset[
            "_qc_status_text"
        ] = (
            candidate_final_subset[
                candidate_status_col
            ]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        candidate_status_counts = {
            status: int(
                (
                    candidate_final_subset[
                        "_qc_status_text"
                    ] == status
                ).sum()
            )
            for status in [
                "PASS",
                "WARN",
                "FAIL",
            ]
        }

        candidate_valid = (
            len(missing_final_ids) == 0
            and
            extra_ids_valid
            and
            len(candidate_final_subset)
            == EXPECTED_INCLUDED_TOTAL
            and
            candidate_status_counts
            == EXPECTED_POOL_QC_STATUS_COUNTS
        )

        qc_candidate_messages.append(
            f"{qc_candidate}: "
            f"总病例={len(candidate_qc_df)}, "
            f"最终子集={len(candidate_final_subset)}, "
            f"缺少={sorted(missing_final_ids)}, "
            f"额外={sorted(extra_qc_ids)}, "
            f"状态={candidate_status_counts}"
        )

        if candidate_valid:
            valid_qc_candidates.append(
                qc_candidate
            )

    except Exception as error:
        qc_candidate_messages.append(
            f"{qc_candidate}: "
            f"读取失败={repr(error)}"
        )

if not valid_qc_candidates:
    raise RuntimeError(
        "没有找到能够覆盖历史预处理候选池497例的有效QC报告。\n\n"
        + "\n".join(
            qc_candidate_messages
        )
    )

QC_REPORT = max(
    valid_qc_candidates,
    key=lambda path: path.stat().st_mtime,
)

print("最终采用QC报告：")
print(" ", QC_REPORT)
print()


# ============================================================
# 10. 正式读取最终QC报告
# ============================================================

qc_df_all = pd.read_excel(
    QC_REPORT,
    sheet_name="Case_level",
    dtype=object,
    engine="openpyxl",
)

qc_df_all.columns = [
    str(column).strip()
    for column
    in qc_df_all.columns
]

required_qc_columns = [
    "patient_id",
    "qc_status",
    "ct_resolved_path",
    "dose_resolved_path",
    "oral_resolved_path",
    "gtv_resolved_path",
    "dose_raw_to_cgy_scale",
    "warning_reasons",
    "fail_reasons",
]

missing_qc_columns = [
    column
    for column
    in required_qc_columns
    if column not in qc_df_all.columns
]

if missing_qc_columns:
    raise KeyError(
        "QC报告Case_level缺少字段："
        f"{missing_qc_columns}"
    )

qc_df_all[
    "patient_id_norm"
] = (
    qc_df_all[
        "patient_id"
    ]
    .map(
        normalize_patient_id
    )
)

qc_df_all[
    "patient_id_int"
] = (
    qc_df_all[
        "patient_id_norm"
    ]
    .astype(int)
)

duplicate_qc_ids = (
    qc_df_all.loc[
        qc_df_all[
            "patient_id_int"
        ].duplicated(
            keep=False
        ),
        "patient_id_int",
    ]
    .astype(int)
    .tolist()
)

if duplicate_qc_ids:
    raise RuntimeError(
        "QC报告存在重复病例编号："
        f"{sorted(set(duplicate_qc_ids))}"
    )

# 仅保留历史预处理候选池497例
qc_df = (
    qc_df_all.loc[
        qc_df_all[
            "patient_id_int"
        ].isin(
            included_ids
        )
    ]
    .copy()
)

qc_df[
    "qc_status_text"
] = (
    qc_df[
        "qc_status"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)

if len(qc_df) != EXPECTED_INCLUDED_TOTAL:
    raise RuntimeError(
        "QC筛选后的病例数不是497："
        f"实际={len(qc_df)}"
    )

qc_ids = set(
    qc_df[
        "patient_id_int"
    ].astype(int)
)

if qc_ids != included_ids:
    raise RuntimeError(
        "QC与最终master病例编号不一致：\n"
        f"QC缺少="
        f"{sorted(included_ids - qc_ids)}\n"
        f"QC多出="
        f"{sorted(qc_ids - included_ids)}"
    )

actual_qc_status_counts = {
    status: int(
        (
            qc_df[
                "qc_status_text"
            ] == status
        ).sum()
    )
    for status in [
        "PASS",
        "WARN",
        "FAIL",
    ]
}

if (
    actual_qc_status_counts
    != EXPECTED_POOL_QC_STATUS_COUNTS
):
    raise RuntimeError(
        "历史预处理候选池497例QC状态不符合预期：\n"
        f"实际={actual_qc_status_counts}\n"
        f"预期="
        f"{EXPECTED_POOL_QC_STATUS_COUNTS}"
    )

actual_source_qc_warning_ids = set(
    qc_df.loc[
        qc_df[
            "qc_status_text"
        ] == "WARN",
        "patient_id_int",
    ]
    .astype(int)
)

if (
    actual_source_qc_warning_ids
    != EXPECTED_SOURCE_QC_WARNING_IDS
):
    raise RuntimeError(
        "最终QC warning病例不符合预期：\n"
        f"实际="
        f"{sorted(actual_source_qc_warning_ids)}\n"
        f"预期="
        f"{sorted(EXPECTED_SOURCE_QC_WARNING_IDS)}"
    )

if (
    qc_df[
        "qc_status_text"
    ] == "FAIL"
).any():
    raise RuntimeError(
        "历史预处理候选池497例QC仍存在FAIL病例"
    )

for path_column in [
    "ct_resolved_path",
    "dose_resolved_path",
    "oral_resolved_path",
    "gtv_resolved_path",
]:
    missing_path_ids = (
        qc_df.loc[
            qc_df[
                path_column
            ].map(
                is_blank
            ),
            "patient_id_int",
        ]
        .astype(int)
        .tolist()
    )

    if missing_path_ids:
        raise RuntimeError(
            f"QC字段{path_column}"
            f"存在空路径："
            f"{missing_path_ids}"
        )

print("历史预处理候选池497例QC验证通过：")
print(f"  PASS：{actual_qc_status_counts['PASS']}")
print(f"  WARN：{actual_qc_status_counts['WARN']}")
print(f"  FAIL：{actual_qc_status_counts['FAIL']}")
print(
    "  已接受的来源QC warning："
    f"{sorted(actual_source_qc_warning_ids)}"
)
print()


# ============================================================
# 11. 合并master与QC
# ============================================================

qc_selected_columns = [
    "patient_id_int",
    "qc_status_text",
    "warning_reasons",
    "fail_reasons",
    "ct_resolved_path",
    "dose_resolved_path",
    "oral_resolved_path",
    "gtv_resolved_path",
    "dose_raw_to_cgy_scale",
]

optional_qc_columns = [
    "dose_inferred_unit",
    "dose_max_cgy",
    "dose_max_to_prescription_ratio",
]

for column in optional_qc_columns:
    if column in qc_df.columns:
        qc_selected_columns.append(
            column
        )

merged_df = included_df.merge(
    qc_df[
        qc_selected_columns
    ],
    on="patient_id_int",
    how="left",
    validate="one_to_one",
)

if len(merged_df) != EXPECTED_INCLUDED_TOTAL:
    raise RuntimeError(
        "master与QC合并后病例数不是497"
    )

merged_df = (
    merged_df
    .sort_values(
        "patient_id_int"
    )
    .reset_index(drop=True)
)

if (
    merged_df[
        "patient_id_int"
    ] == PREPROCESSING_EXCLUDED_ID
).any():
    raise RuntimeError(
        "严重错误：配置的排除病例进入了合并表"
    )


# ============================================================
# 12. 初始化正式输出目录
# ============================================================

if OUTPUT_ROOT.exists():
    if not ALLOW_OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"输出目录已经存在：\n"
            f"{OUTPUT_ROOT}\n\n"
            "为防止覆盖正式结果，程序已停止。\n"
            "若确实需要完全重建：\n"
            "1. 删除这个目录；或\n"
            "2. 将OUTPUT_VERSION改为v3。"
        )

    shutil.rmtree(
        OUTPUT_ROOT
    )

NPZ_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

OVERLAY_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

master_sha256 = sha256_file(
    MASTER_XLSX
)

qc_sha256 = sha256_file(
    QC_REPORT
)

shutil.copy2(
    MASTER_XLSX,
    OUTPUT_ROOT
    / "master_frozen_used.xlsx",
)

shutil.copy2(
    QC_REPORT,
    OUTPUT_ROOT
    / "source_QC_report_used.xlsx",
)

config = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),

    "master_xlsx": str(
        MASTER_XLSX
    ),

    "master_sha256": (
        master_sha256
    ),

    "expected_master_sha256": (
        EXPECTED_MASTER_SHA256
    ),

    "strict_master_sha256": (
        STRICT_MASTER_SHA256
    ),

    "source_qc_report": str(
        QC_REPORT
    ),

    "source_qc_sha256": (
        qc_sha256
    ),

    "source_qc_total_rows": (
        len(qc_df_all)
    ),

    "final_qc_subset_rows": (
        len(qc_df)
    ),

    "preprocessing_excluded_patient_id": (
        PREPROCESSING_EXCLUDED_ID
    ),

    "output_root": str(
        OUTPUT_ROOT
    ),

    "expected_master_total": (
        EXPECTED_MASTER_TOTAL
    ),

    "expected_excluded_total": (
        EXPECTED_EXCLUDED_TOTAL
    ),

    "expected_total": (
        EXPECTED_INCLUDED_TOTAL
    ),

    "expected_development": 269,

    "expected_external": 228,

    "model_center_counts": (
        EXPECTED_MODEL_CENTER_COUNTS
    ),

    "model_center_label_counts": {
        f"{center}_{label}": count
        for (
            center,
            label
        ), count
        in EXPECTED_MODEL_CENTER_LABEL_COUNTS.items()
    },

    "total_label_counts": (
        EXPECTED_TOTAL_LABEL_COUNTS
    ),

    "target_spacing_xyz_mm": (
        TARGET_SPACING_XYZ
    ),

    "patch_size_xyz_voxels": (
        PATCH_SIZE_XYZ
    ),

    "array_shape_zyx": (
        EXPECTED_ARRAY_SHAPE_ZYX
    ),

    "crop_center": (
        "oral_cavity_bounding_box_physical_center"
    ),

    "ct_interpolation": "linear",

    "dose_interpolation": "linear",

    "mask_interpolation": (
        "nearest_neighbor"
    ),

    "ct_default_value_hu": (
        CT_CLIP_MIN_HU
    ),

    "ct_clip_hu": [
        CT_CLIP_MIN_HU,
        CT_CLIP_MAX_HU,
    ],

    "ct_normalization": (
        "linear_to_minus1_plus1"
    ),

    "dose_default_value": 0.0,

    "dose_unit_after_conversion": "Gy",

    "dose_normalization_gy": (
        DOSE_NORMALIZATION_GY
    ),

    "dose_normalized_clip": [
        0.0,
        DOSE_NORMALIZED_MAX,
    ],

    "oral_retention_required": (
        ORAL_RETENTION_REQUIRED
    ),

    "gtv_retention_warning": (
        GTV_RETENTION_WARNING
    ),

    "expected_source_qc_warning_ids": (
        sorted(
            EXPECTED_SOURCE_QC_WARNING_IDS
        )
    ),

    "expected_preprocessing_warning_ids": (
        sorted(
            EXPECTED_PREPROCESSING_WARNING_IDS
        )
    ),

    "save_all_overlays": (
        SAVE_ALL_OVERLAYS
    ),
}

CONFIG_JSON.write_text(
    json.dumps(
        config,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# 13. 批量生成497例NPZ
# ============================================================

case_records = []

for _, row in tqdm(
    merged_df.iterrows(),
    total=len(merged_df),
    desc="Generating 497 NPZ files",
):
    patient_id = int(
        row[
            "patient_id_int"
        ]
    )

    if patient_id == PREPROCESSING_EXCLUDED_ID:
        raise RuntimeError(
            "严重错误：循环中发现配置的排除病例"
        )

    original_center = int(
        row[
            "original_center_int"
        ]
    )

    model_center = str(
        row[
            "model_center_text"
        ]
    )

    cohort = str(
        row[
            "cohort_text"
        ]
    )

    label = int(
        row[
            "label_int"
        ]
    )

    source_qc_status = str(
        row[
            "qc_status_text"
        ]
    )

    source_qc_warning = (
        ""
        if is_blank(
            row[
                "warning_reasons"
            ]
        )
        else
        str(
            row[
                "warning_reasons"
            ]
        )
    )

    record = {
        "patient_id": patient_id,
        "original_center": original_center,
        "model_center": model_center,
        "cohort": cohort,
        "severe_mucositis": label,

        "source_qc_status": (
            source_qc_status
        ),

        "source_qc_warning": (
            source_qc_warning
        ),

        "status": "failed",
        "warnings": "",
        "error": "",
    }

    try:
        ct_path = Path(
            str(
                row[
                    "ct_resolved_path"
                ]
            )
        )

        dose_path = Path(
            str(
                row[
                    "dose_resolved_path"
                ]
            )
        )

        oral_path = Path(
            str(
                row[
                    "oral_resolved_path"
                ]
            )
        )

        gtv_path = Path(
            str(
                row[
                    "gtv_resolved_path"
                ]
            )
        )

        record.update({
            "ct_path": str(
                ct_path
            ),

            "dose_path": str(
                dose_path
            ),

            "oral_path": str(
                oral_path
            ),

            "gtv_path": str(
                gtv_path
            ),
        })

        for modality, path in [
            ("ct", ct_path),
            ("dose", dose_path),
            ("oral", oral_path),
            ("gtv", gtv_path),
        ]:
            if not path.exists():
                raise FileNotFoundError(
                    f"{modality}文件不存在："
                    f"{path}"
                )

        ct_image = read_3d_image(
            ct_path
        )

        dose_image = read_3d_image(
            dose_path
        )

        oral_image = read_3d_image(
            oral_path
        )

        gtv_image = read_3d_image(
            gtv_path
        )

        record.update({
            "original_size_x": (
                ct_image.GetSize()[0]
            ),

            "original_size_y": (
                ct_image.GetSize()[1]
            ),

            "original_size_z": (
                ct_image.GetSize()[2]
            ),

            "original_spacing_x": (
                ct_image.GetSpacing()[0]
            ),

            "original_spacing_y": (
                ct_image.GetSpacing()[1]
            ),

            "original_spacing_z": (
                ct_image.GetSpacing()[2]
            ),
        })

        geometry_failures = []

        for modality, image in [
            ("dose", dose_image),
            ("oral", oral_image),
            ("gtv", gtv_image),
        ]:
            details = geometry_detail(
                ct_image,
                image,
            )

            record[
                f"{modality}_vs_ct_size_equal"
            ] = details[
                "size_equal"
            ]

            record[
                f"{modality}_vs_ct_max_spacing_diff_mm"
            ] = details[
                "max_spacing_diff_mm"
            ]

            record[
                f"{modality}_vs_ct_max_origin_diff_mm"
            ] = details[
                "max_origin_diff_mm"
            ]

            record[
                f"{modality}_vs_ct_max_direction_diff"
            ] = details[
                "max_direction_diff"
            ]

            if not same_geometry(
                ct_image,
                image,
            ):
                geometry_failures.append(
                    modality
                )

        if geometry_failures:
            raise RuntimeError(
                "进入NPZ预处理前发现几何不一致："
                f"{geometry_failures}"
            )

        oral_original = sitk.Cast(
            oral_image > 0,
            sitk.sitkUInt8,
        )

        gtv_original = sitk.Cast(
            gtv_image > 0,
            sitk.sitkUInt8,
        )

        if int(
            binary_array(
                oral_original
            ).sum()
        ) <= 0:
            raise RuntimeError(
                "原始oral mask为空"
            )

        if int(
            binary_array(
                gtv_original
            ).sum()
        ) <= 0:
            raise RuntimeError(
                "原始GTV mask为空"
            )

        (
            oral_center_physical_xyz,
            oral_center_index_xyz,
        ) = mask_bbox_center_physical(
            oral_original
        )

        patch_reference = (
            create_patch_reference(
                ct_image,
                oral_center_physical_xyz,
            )
        )

        oral_retained_fraction = (
            patch_retained_fraction(
                oral_original,
                patch_reference,
            )
        )

        gtv_retained_fraction = (
            patch_retained_fraction(
                gtv_original,
                patch_reference,
            )
        )

        original_oral_volume_cc = (
            physical_volume_cc(
                oral_original
            )
        )

        original_gtv_volume_cc = (
            physical_volume_cc(
                gtv_original
            )
        )

        record.update({
            "oral_center_index_x": (
                oral_center_index_xyz[0]
            ),

            "oral_center_index_y": (
                oral_center_index_xyz[1]
            ),

            "oral_center_index_z": (
                oral_center_index_xyz[2]
            ),

            "oral_center_physical_x": (
                oral_center_physical_xyz[0]
            ),

            "oral_center_physical_y": (
                oral_center_physical_xyz[1]
            ),

            "oral_center_physical_z": (
                oral_center_physical_xyz[2]
            ),

            "patch_origin_x": (
                patch_reference.GetOrigin()[0]
            ),

            "patch_origin_y": (
                patch_reference.GetOrigin()[1]
            ),

            "patch_origin_z": (
                patch_reference.GetOrigin()[2]
            ),

            "oral_retained_fraction": (
                oral_retained_fraction
            ),

            "gtv_retained_fraction": (
                gtv_retained_fraction
            ),

            "original_oral_volume_cc": (
                original_oral_volume_cc
            ),

            "original_gtv_volume_cc": (
                original_gtv_volume_cc
            ),
        })

        if not np.isfinite(
            oral_retained_fraction
        ):
            raise RuntimeError(
                "无法计算oral保留比例"
            )

        if (
            oral_retained_fraction
            < ORAL_RETENTION_REQUIRED
        ):
            raise RuntimeError(
                "oral被固定patch裁剪："
                f"保留比例="
                f"{oral_retained_fraction:.6f}"
            )

        preprocessing_warnings = []

        if source_qc_status == "WARN":
            preprocessing_warnings.append(
                "accepted_source_QC_warning:"
                f"{source_qc_warning}"
            )

        if (
            np.isfinite(
                gtv_retained_fraction
            )
            and
            gtv_retained_fraction
            < GTV_RETENTION_WARNING
        ):
            preprocessing_warnings.append(
                "gtv_retained_fraction="
                f"{gtv_retained_fraction:.6f}"
            )

        (
            dose_scale_to_gy,
            dose_scale_source,
        ) = infer_dose_scale_to_gy(
            dose_image,
            row[
                "dose_raw_to_cgy_scale"
            ],
        )

        # ----------------------------------------------------
        # 每个模态仅重采样一次
        # ----------------------------------------------------

        ct_patch = resample_to_reference(
            ct_image,
            patch_reference,
            sitk.sitkLinear,
            CT_CLIP_MIN_HU,
            sitk.sitkFloat32,
        )

        dose_patch = resample_to_reference(
            dose_image,
            patch_reference,
            sitk.sitkLinear,
            0.0,
            sitk.sitkFloat32,
        )

        oral_patch = resample_to_reference(
            oral_original,
            patch_reference,
            sitk.sitkNearestNeighbor,
            0,
            sitk.sitkUInt8,
        )

        gtv_patch = resample_to_reference(
            gtv_original,
            patch_reference,
            sitk.sitkNearestNeighbor,
            0,
            sitk.sitkUInt8,
        )

        arrays = prepare_arrays(
            ct_patch,
            dose_patch,
            oral_patch,
            gtv_patch,
            dose_scale_to_gy,
        )

        output_oral_volume_cc = float(
            arrays[
                "oral"
            ].sum()
            * np.prod(
                TARGET_SPACING_XYZ
            )
            / 1000.0
        )

        output_gtv_volume_cc = float(
            arrays[
                "gtv"
            ].sum()
            * np.prod(
                TARGET_SPACING_XYZ
            )
            / 1000.0
        )

        oral_volume_ratio = (
            output_oral_volume_cc
            / original_oral_volume_cc
            if original_oral_volume_cc > 0
            else np.nan
        )

        gtv_volume_ratio = (
            output_gtv_volume_cc
            / original_gtv_volume_cc
            if original_gtv_volume_cc > 0
            else np.nan
        )

        dose_patch_max_gy = float(
            arrays[
                "dose_gy_for_qc"
            ].max()
        )

        oral_dose_values = (
            arrays[
                "dose_gy_for_qc"
            ][
                arrays["oral"] > 0
            ]
        )

        gtv_dose_values = (
            arrays[
                "dose_gy_for_qc"
            ][
                arrays["gtv"] > 0
            ]
        )

        dose_patch_mean_oral_gy = (
            float(
                oral_dose_values.mean()
            )
            if oral_dose_values.size > 0
            else np.nan
        )

        dose_patch_mean_gtv_gy = (
            float(
                gtv_dose_values.mean()
            )
            if gtv_dose_values.size > 0
            else np.nan
        )

        prescription_dose_cgy = (
            require_float(
                row[
                    MASTER_PRESCRIPTION_COL
                ],
                MASTER_PRESCRIPTION_COL,
                patient_id,
            )
        )

        fraction_dose_cgy = (
            require_float(
                row[
                    MASTER_FRACTION_DOSE_COL
                ],
                MASTER_FRACTION_DOSE_COL,
                patient_id,
            )
        )

        fractions = require_int(
            row[
                MASTER_FRACTIONS_COL
            ],
            MASTER_FRACTIONS_COL,
            patient_id,
        )

        rt_start_year = require_int(
            row[
                MASTER_RT_YEAR_COL
            ],
            MASTER_RT_YEAR_COL,
            patient_id,
        )

        rt_technique = normalize_text(
            row[
                MASTER_RT_TECHNIQUE_COL
            ]
        )

        if rt_technique == "":
            raise ValueError(
                f"病例{patient_id}"
                "的RT_technique为空"
            )

        metadata = {
            "patient_id": (
                patient_id
            ),

            "original_center": (
                original_center
            ),

            "model_center": (
                model_center
            ),

            "cohort": (
                cohort
            ),

            "label": (
                label
            ),

            "prescription_dose_cgy": (
                prescription_dose_cgy
            ),

            "fraction_dose_cgy": (
                fraction_dose_cgy
            ),

            "fractions": (
                fractions
            ),

            "rt_technique": (
                rt_technique
            ),

            "rt_start_year": (
                rt_start_year
            ),

            "source_qc_status": (
                source_qc_status
            ),

            "source_qc_warning": (
                source_qc_warning
            ),

            "patch_origin_xyz": (
                patch_reference.GetOrigin()
            ),

            "patch_direction": (
                patch_reference.GetDirection()
            ),

            "oral_center_physical_xyz": (
                oral_center_physical_xyz
            ),

            "oral_center_index_xyz": (
                oral_center_index_xyz
            ),

            "dose_scale_to_gy": (
                dose_scale_to_gy
            ),
        }

        output_npz_path = (
            NPZ_DIR
            / f"{patient_id}.npz"
        )

        save_npz_atomic(
            output_npz_path,
            arrays,
            metadata,
        )

        overlay_saved = False
        overlay_path = ""

        if (
            SAVE_ALL_OVERLAYS
            or preprocessing_warnings
        ):
            overlay_output_path = (
                OVERLAY_DIR
                / f"{patient_id}_NPZ_QC.png"
            )

            save_combined_overlay(
                arrays=arrays,
                patient_id=patient_id,
                output_path=overlay_output_path,
                status_text=(
                    " | ".join(
                        preprocessing_warnings
                    )
                    if preprocessing_warnings
                    else "PASS"
                ),
            )

            overlay_saved = True

            overlay_path = str(
                overlay_output_path
            )

        record.update({
            "dose_scale_to_Gy": (
                dose_scale_to_gy
            ),

            "dose_scale_source": (
                dose_scale_source
            ),

            "output_npz_path": str(
                output_npz_path
            ),

            "output_shape_z": (
                arrays["ct"].shape[0]
            ),

            "output_shape_y": (
                arrays["ct"].shape[1]
            ),

            "output_shape_x": (
                arrays["ct"].shape[2]
            ),

            "output_oral_voxels": int(
                arrays["oral"].sum()
            ),

            "output_gtv_voxels": int(
                arrays["gtv"].sum()
            ),

            "output_oral_volume_cc": (
                output_oral_volume_cc
            ),

            "output_gtv_volume_cc": (
                output_gtv_volume_cc
            ),

            "oral_volume_ratio_after_resampling": (
                oral_volume_ratio
            ),

            "gtv_volume_ratio_after_resampling": (
                gtv_volume_ratio
            ),

            "ct_normalized_min": float(
                arrays["ct"].min()
            ),

            "ct_normalized_max": float(
                arrays["ct"].max()
            ),

            "dose_normalized_min": float(
                arrays["dose"].min()
            ),

            "dose_normalized_max": float(
                arrays["dose"].max()
            ),

            "dose_patch_max_Gy": (
                dose_patch_max_gy
            ),

            "dose_patch_mean_oral_Gy": (
                dose_patch_mean_oral_gy
            ),

            "dose_patch_mean_gtv_Gy": (
                dose_patch_mean_gtv_gy
            ),

            "warnings": (
                " | ".join(
                    preprocessing_warnings
                )
            ),

            "status": (
                "success_with_warning"
                if preprocessing_warnings
                else "success"
            ),

            "overlay_saved": (
                overlay_saved
            ),

            "overlay_path": (
                overlay_path
            ),
        })

    except Exception:
        record[
            "error"
        ] = traceback.format_exc()

    case_records.append(
        record
    )

    # 主动释放内存
    for variable_name in [
        "ct_image",
        "dose_image",
        "oral_image",
        "gtv_image",
        "oral_original",
        "gtv_original",
        "patch_reference",
        "ct_patch",
        "dose_patch",
        "oral_patch",
        "gtv_patch",
        "arrays",
    ]:
        if variable_name in locals():
            try:
                del locals()[
                    variable_name
                ]
            except Exception:
                pass

    gc.collect()


# ============================================================
# 14. 整理预处理结果
# ============================================================

manifest_df = (
    pd.DataFrame(
        case_records
    )
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

success_df = manifest_df.loc[
    manifest_df[
        "status"
    ].isin(
        [
            "success",
            "success_with_warning",
        ]
    )
].copy()

failure_df = manifest_df.loc[
    ~manifest_df[
        "status"
    ].isin(
        [
            "success",
            "success_with_warning",
        ]
    )
].copy()

warning_df = manifest_df.loc[
    manifest_df[
        "status"
    ] == "success_with_warning"
].copy()


# ============================================================
# 15. 重新打开全部NPZ执行审计
# ============================================================

required_npz_keys = {
    "ct",
    "dose",
    "oral",
    "gtv",

    "patient_id",
    "center",
    "original_center",
    "model_center",
    "cohort",
    "label",
    "severe_mucositis",

    "target_spacing_xyz",
    "patch_size_xyz",
    "patch_origin_xyz",
    "patch_direction",

    "oral_center_physical_xyz",
    "oral_center_index_xyz",

    "ct_clip_hu",
    "dose_normalization_gy",
    "dose_normalized_max",
    "dose_scale_to_gy",
}

audit_records = []

expected_ids = set(
    merged_df[
        "patient_id_int"
    ].astype(int)
)

actual_npz_paths = sorted(
    NPZ_DIR.glob(
        "*.npz"
    ),
    key=lambda path: int(
        path.stem
    ),
)

actual_npz_ids = {
    int(path.stem)
    for path
    in actual_npz_paths
}

missing_npz_ids = sorted(
    expected_ids
    - actual_npz_ids
)

extra_npz_ids = sorted(
    actual_npz_ids
    - expected_ids
)

if PREPROCESSING_EXCLUDED_ID in actual_npz_ids:
    raise RuntimeError(
        "严重错误：NPZ目录中出现了配置的排除病例NPZ"
    )

for npz_path in tqdm(
    actual_npz_paths,
    desc="Auditing generated 497 NPZ files",
):
    patient_id_from_filename = int(
        npz_path.stem
    )

    audit_record = {
        "patient_id": (
            patient_id_from_filename
        ),

        "npz_path": str(
            npz_path
        ),

        "audit_status": "failed",
        "audit_errors": "",
    }

    audit_errors = []

    try:
        with np.load(
            npz_path,
            allow_pickle=False,
        ) as data:

            available_keys = set(
                data.files
            )

            missing_keys = (
                required_npz_keys
                - available_keys
            )

            if missing_keys:
                audit_errors.append(
                    "missing_keys="
                    f"{sorted(missing_keys)}"
                )

            for key in [
                "ct",
                "dose",
                "oral",
                "gtv",
            ]:
                if key not in data:
                    continue

                array = data[key]

                audit_record[
                    f"{key}_shape"
                ] = str(
                    tuple(
                        array.shape
                    )
                )

                audit_record[
                    f"{key}_dtype"
                ] = str(
                    array.dtype
                )

                if (
                    tuple(array.shape)
                    != EXPECTED_ARRAY_SHAPE_ZYX
                ):
                    audit_errors.append(
                        f"{key}_shape="
                        f"{array.shape}"
                    )

            if "ct" in data:
                ct_array = data["ct"]

                if (
                    ct_array.dtype
                    != np.float32
                ):
                    audit_errors.append(
                        "ct_dtype_not_float32"
                    )

                if not np.isfinite(
                    ct_array
                ).all():
                    audit_errors.append(
                        "ct_has_nan_or_inf"
                    )

                if (
                    float(
                        ct_array.min()
                    ) < -1.0001
                    or
                    float(
                        ct_array.max()
                    ) > 1.0001
                ):
                    audit_errors.append(
                        "ct_range_invalid"
                    )

                audit_record[
                    "ct_min"
                ] = float(
                    ct_array.min()
                )

                audit_record[
                    "ct_max"
                ] = float(
                    ct_array.max()
                )

            if "dose" in data:
                dose_array = data[
                    "dose"
                ]

                if (
                    dose_array.dtype
                    != np.float32
                ):
                    audit_errors.append(
                        "dose_dtype_not_float32"
                    )

                if not np.isfinite(
                    dose_array
                ).all():
                    audit_errors.append(
                        "dose_has_nan_or_inf"
                    )

                if (
                    float(
                        dose_array.min()
                    ) < -1e-6
                    or
                    float(
                        dose_array.max()
                    )
                    > DOSE_NORMALIZED_MAX + 1e-6
                ):
                    audit_errors.append(
                        "dose_range_invalid"
                    )

                audit_record[
                    "dose_min"
                ] = float(
                    dose_array.min()
                )

                audit_record[
                    "dose_max"
                ] = float(
                    dose_array.max()
                )

            for mask_key in [
                "oral",
                "gtv",
            ]:
                if mask_key not in data:
                    continue

                mask_array = data[
                    mask_key
                ]

                if (
                    mask_array.dtype
                    != np.uint8
                ):
                    audit_errors.append(
                        f"{mask_key}_dtype_not_uint8"
                    )

                unique_values = set(
                    np.unique(
                        mask_array
                    ).tolist()
                )

                if not unique_values.issubset(
                    {
                        0,
                        1,
                    }
                ):
                    audit_errors.append(
                        f"{mask_key}_not_binary:"
                        f"{sorted(unique_values)}"
                    )

                voxel_count = int(
                    mask_array.sum()
                )

                audit_record[
                    f"{mask_key}_voxels"
                ] = voxel_count

                if voxel_count <= 0:
                    audit_errors.append(
                        f"{mask_key}_empty"
                    )

            if "patient_id" in data:
                patient_id_inside = int(
                    np.asarray(
                        data[
                            "patient_id"
                        ]
                    ).item()
                )

                audit_record[
                    "patient_id_inside"
                ] = patient_id_inside

                if (
                    patient_id_inside
                    != patient_id_from_filename
                ):
                    audit_errors.append(
                        "patient_id_mismatch"
                    )

                if (
                    patient_id_inside
                    == PREPROCESSING_EXCLUDED_ID
                ):
                    audit_errors.append(
                        "configured_preprocessing_excluded_present"
                    )

            if "label" in data:
                label_inside = int(
                    np.asarray(
                        data["label"]
                    ).item()
                )

                audit_record[
                    "label"
                ] = label_inside

                if label_inside not in {
                    0,
                    1,
                }:
                    audit_errors.append(
                        "invalid_label"
                    )

            if (
                "target_spacing_xyz"
                in data
            ):
                spacing_inside = np.asarray(
                    data[
                        "target_spacing_xyz"
                    ],
                    dtype=float,
                )

                if not np.allclose(
                    spacing_inside,
                    TARGET_SPACING_XYZ,
                    atol=1e-6,
                    rtol=0,
                ):
                    audit_errors.append(
                        "target_spacing_mismatch"
                    )

            if "patch_size_xyz" in data:
                size_inside = tuple(
                    np.asarray(
                        data[
                            "patch_size_xyz"
                        ],
                        dtype=int,
                    ).tolist()
                )

                if (
                    size_inside
                    != PATCH_SIZE_XYZ
                ):
                    audit_errors.append(
                        "patch_size_mismatch"
                    )

            if "model_center" in data:
                audit_record[
                    "model_center"
                ] = scalar_text(
                    data[
                        "model_center"
                    ]
                )

            if "cohort" in data:
                audit_record[
                    "cohort"
                ] = scalar_text(
                    data[
                        "cohort"
                    ]
                )

        audit_record[
            "audit_status"
        ] = (
            "pass"
            if not audit_errors
            else "failed"
        )

        audit_record[
            "audit_errors"
        ] = " | ".join(
            audit_errors
        )

    except Exception:
        audit_record[
            "audit_errors"
        ] = traceback.format_exc()

    audit_records.append(
        audit_record
    )

audit_df = (
    pd.DataFrame(
        audit_records
    )
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

audit_failure_df = (
    audit_df.loc[
        audit_df[
            "audit_status"
        ] != "pass"
    ]
    .copy()
)


# ============================================================
# 16. 生成最终dataset index
# ============================================================

output_metadata_columns = [
    "patient_id",
    "output_npz_path",

    "oral_retained_fraction",
    "gtv_retained_fraction",

    "original_oral_volume_cc",
    "original_gtv_volume_cc",

    "output_oral_volume_cc",
    "output_gtv_volume_cc",

    "dose_scale_to_Gy",

    "dose_patch_max_Gy",
    "dose_patch_mean_oral_Gy",
    "dose_patch_mean_gtv_Gy",

    "warnings",
    "status",
]

index_source_df = (
    success_df[
        output_metadata_columns
    ]
    .rename(
        columns={
            "patient_id":
            "patient_id_int"
        }
    )
)

dataset_index_df = (
    included_df.merge(
        index_source_df,
        on="patient_id_int",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "patient_id_int"
    )
    .reset_index(
        drop=True
    )
)

internal_columns_to_drop = [
    "patient_id_norm",
    "original_center_int",
    "model_center_text",
    "cohort_text",
    "label_int",
]

dataset_index_df = (
    dataset_index_df.drop(
        columns=[
            column
            for column
            in internal_columns_to_drop
            if column
            in dataset_index_df.columns
        ]
    )
)

if len(dataset_index_df) != EXPECTED_INCLUDED_TOTAL:
    raise RuntimeError(
        "dataset_index不是497行："
        f"{len(dataset_index_df)}"
    )

if (
    dataset_index_df[
        "patient_id_int"
    ].astype(int)
    == PREPROCESSING_EXCLUDED_ID
).any():
    raise RuntimeError(
        "dataset_index中仍存在配置的排除病例"
    )

dataset_index_df.to_csv(
    INDEX_CSV,
    index=False,
    encoding="utf-8-sig",
)

audit_df.to_csv(
    AUDIT_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 17. 生成汇总报告
# ============================================================

success_center_counts = (
    success_df[
        "model_center"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    if len(success_df)
    else {}
)

success_cohort_counts = (
    success_df[
        "cohort"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    if len(success_df)
    else {}
)

success_label_counts = (
    success_df[
        "severe_mucositis"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
    if len(success_df)
    else {}
)

actual_preprocessing_warning_ids = set(
    warning_df[
        "patient_id"
    ].astype(int)
)

summary_rows = [
    [
        "master_total_rows",
        len(master_df),
    ],

    [
        "master_excluded_rows",
        len(excluded_df),
    ],

    [
        "requested_final_cases",
        len(merged_df),
    ],

    [
        "successful_npz_cases",
        len(success_df),
    ],

    [
        "failed_preprocessing_cases",
        len(failure_df),
    ],

    [
        "preprocessing_warning_cases",
        len(warning_df),
    ],

    [
        "preprocessing_warning_ids",
        ",".join(
            str(value)
            for value
            in sorted(
                actual_preprocessing_warning_ids
            )
        ),
    ],

    [
        "generated_npz_files",
        len(actual_npz_paths),
    ],

    [
        "missing_npz_ids",
        ",".join(
            str(value)
            for value
            in missing_npz_ids
        ),
    ],

    [
        "extra_npz_ids",
        ",".join(
            str(value)
            for value
            in extra_npz_ids
        ),
    ],

    [
        "configured_excluded_case_present_in_npz",
        PREPROCESSING_EXCLUDED_ID
        in actual_npz_ids,
    ],

    [
        "npz_audit_pass",
        int(
            (
                audit_df[
                    "audit_status"
                ] == "pass"
            ).sum()
        ),
    ],

    [
        "npz_audit_failed",
        len(
            audit_failure_df
        ),
    ],

    [
        "development_cases",
        success_cohort_counts.get(
            "Development",
            0,
        ),
    ],

    [
        "external_cases",
        success_cohort_counts.get(
            "External",
            0,
        ),
    ],

    [
        "center_A_cases",
        success_center_counts.get(
            "A",
            0,
        ),
    ],

    [
        "center_B_cases",
        success_center_counts.get(
            "B",
            0,
        ),
    ],

    [
        "center_C_cases",
        success_center_counts.get(
            "C",
            0,
        ),
    ],

    [
        "nonsevere_cases",
        success_label_counts.get(
            0,
            0,
        ),
    ],

    [
        "severe_cases",
        success_label_counts.get(
            1,
            0,
        ),
    ],

    [
        "oral_retention_min",
        float(
            success_df[
                "oral_retained_fraction"
            ].min()
        )
        if len(success_df)
        else np.nan,
    ],

    [
        "gtv_retention_min",
        float(
            success_df[
                "gtv_retained_fraction"
            ].min()
        )
        if len(success_df)
        else np.nan,
    ],

    [
        "master_sha256",
        master_sha256,
    ],

    [
        "source_QC_sha256",
        qc_sha256,
    ],
]

summary_df = pd.DataFrame(
    summary_rows,
    columns=[
        "metric",
        "value",
    ],
)

config_df = pd.DataFrame(
    [
        [
            key,
            json.dumps(
                value,
                ensure_ascii=False,
                default=str,
            ),
        ]
        for key, value
        in config.items()
    ],
    columns=[
        "parameter",
        "value",
    ],
)

with pd.ExcelWriter(
    REPORT_XLSX,
    engine="openpyxl",
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False,
    )

    config_df.to_excel(
        writer,
        sheet_name="Config",
        index=False,
    )

    manifest_df.to_excel(
        writer,
        sheet_name="Manifest",
        index=False,
    )

    success_df.to_excel(
        writer,
        sheet_name="Successful",
        index=False,
    )

    warning_df.to_excel(
        writer,
        sheet_name="Warnings",
        index=False,
    )

    failure_df.to_excel(
        writer,
        sheet_name="Failures",
        index=False,
    )

    audit_df.to_excel(
        writer,
        sheet_name="Audit",
        index=False,
    )

    audit_failure_df.to_excel(
        writer,
        sheet_name="Audit_failures",
        index=False,
    )

    dataset_index_df.to_excel(
        writer,
        sheet_name="Dataset_index",
        index=False,
    )

format_excel_report(
    REPORT_XLSX
)


# ============================================================
# 18. 保存运行日志
# ============================================================

log_lines = [
    "=" * 80,

    "Created at: "
    + datetime.now().isoformat(
        timespec="seconds"
    ),

    f"Master: {MASTER_XLSX}",
    f"Master SHA256: {master_sha256}",

    f"Source QC: {QC_REPORT}",
    f"Source QC SHA256: {qc_sha256}",

    f"Output root: {OUTPUT_ROOT}",

    f"Historical preprocessing-only excluded ID: {PREPROCESSING_EXCLUDED_ID}",

    f"Successful NPZ cases: {len(success_df)}",
    f"Failed preprocessing cases: {len(failure_df)}",
    f"Warning cases: {len(warning_df)}",

    "Warning IDs: "
    + str(
        sorted(
            actual_preprocessing_warning_ids
        )
    ),

    f"Generated NPZ files: {len(actual_npz_paths)}",

    f"Missing NPZ IDs: {missing_npz_ids}",
    f"Extra NPZ IDs: {extra_npz_ids}",

    "Configured excluded case present in NPZ: "
    + str(
        PREPROCESSING_EXCLUDED_ID
        in actual_npz_ids
    ),

    "Audit passed: "
    + str(
        int(
            (
                audit_df[
                    "audit_status"
                ] == "pass"
            ).sum()
        )
    ),

    f"Audit failed: {len(audit_failure_df)}",

    "Development: "
    + str(
        success_cohort_counts.get(
            "Development",
            0,
        )
    ),

    "External: "
    + str(
        success_cohort_counts.get(
            "External",
            0,
        )
    ),

    "Center A: "
    + str(
        success_center_counts.get(
            "A",
            0,
        )
    ),

    "Center B: "
    + str(
        success_center_counts.get(
            "B",
            0,
        )
    ),

    "Center C: "
    + str(
        success_center_counts.get(
            "C",
            0,
        )
    ),

    "Non-severe: "
    + str(
        success_label_counts.get(
            0,
            0,
        )
    ),

    "Severe: "
    + str(
        success_label_counts.get(
            1,
            0,
        )
    ),

    "Target spacing XYZ: "
    + str(
        TARGET_SPACING_XYZ
    ),

    "Patch size XYZ: "
    + str(
        PATCH_SIZE_XYZ
    ),

    "Array shape ZYX: "
    + str(
        EXPECTED_ARRAY_SHAPE_ZYX
    ),
]

RUN_LOG.write_text(
    "\n".join(
        log_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 19. 最终严格完整性检查
# ============================================================

print()
print("=" * 88)
print("历史预处理候选池497例NPZ生成与审计完成")
print("=" * 88)

print(f"请求病例数：{len(merged_df)}")
print(f"成功生成：{len(success_df)}")
print(f"预处理失败：{len(failure_df)}")
print(f"预处理警告：{len(warning_df)}")
print(
    "预处理warning病例："
    f"{sorted(actual_preprocessing_warning_ids)}"
)
print(f"NPZ文件数：{len(actual_npz_paths)}")
print(
    "NPZ审计通过："
    f"{int((audit_df['audit_status'] == 'pass').sum())}"
)
print(
    "NPZ审计失败："
    f"{len(audit_failure_df)}"
)
print()

print("中心与队列：")
print(
    "  Center A："
    f"{success_center_counts.get('A', 0)}"
)
print(
    "  Center B："
    f"{success_center_counts.get('B', 0)}"
)
print(
    "  Center C："
    f"{success_center_counts.get('C', 0)}"
)
print(
    "  Development："
    f"{success_cohort_counts.get('Development', 0)}"
)
print(
    "  External："
    f"{success_cohort_counts.get('External', 0)}"
)
print(
    "  Non-severe："
    f"{success_label_counts.get(0, 0)}"
)
print(
    "  Severe："
    f"{success_label_counts.get(1, 0)}"
)
print()

print("主要输出：")
print("  NPZ目录：", NPZ_DIR)
print("  裁剪后叠加图：", OVERLAY_DIR)
print("  完整报告：", REPORT_XLSX)
print("  数据索引：", INDEX_CSV)
print("  NPZ审计：", AUDIT_CSV)
print("  配置文件：", CONFIG_JSON)
print("  运行日志：", RUN_LOG)
print()

final_errors = []

if len(failure_df) > 0:
    final_errors.append(
        f"有{len(failure_df)}例预处理失败"
    )

if len(success_df) != EXPECTED_INCLUDED_TOTAL:
    final_errors.append(
        "成功生成病例数不是497："
        f"{len(success_df)}"
    )

if len(actual_npz_paths) != EXPECTED_INCLUDED_TOTAL:
    final_errors.append(
        "NPZ文件数不是497："
        f"{len(actual_npz_paths)}"
    )

if missing_npz_ids:
    final_errors.append(
        f"缺失NPZ病例：{missing_npz_ids}"
    )

if extra_npz_ids:
    final_errors.append(
        f"多余NPZ病例：{extra_npz_ids}"
    )

if PREPROCESSING_EXCLUDED_ID in actual_npz_ids:
    final_errors.append(
        "历史预处理阶段配置的排除病例仍存在于NPZ目录"
    )

if len(audit_failure_df) > 0:
    final_errors.append(
        "NPZ审计失败病例："
        + str(
            audit_failure_df[
                "patient_id"
            ].tolist()
        )
    )

if (
    success_center_counts
    != EXPECTED_MODEL_CENTER_COUNTS
):
    final_errors.append(
        "历史预处理候选池中心数量错误："
        f"{success_center_counts}"
    )

if (
    success_cohort_counts
    != EXPECTED_COHORT_COUNTS
):
    final_errors.append(
        "历史预处理候选池队列元数据数量错误："
        f"{success_cohort_counts}"
    )

if (
    success_label_counts
    != EXPECTED_TOTAL_LABEL_COUNTS
):
    final_errors.append(
        "历史预处理候选池标签数量错误："
        f"{success_label_counts}"
    )

if (
    actual_preprocessing_warning_ids
    != EXPECTED_PREPROCESSING_WARNING_IDS
):
    final_errors.append(
        "预处理warning病例与预期不一致："
        f"实际="
        f"{sorted(actual_preprocessing_warning_ids)}；"
        f"预期="
        f"{sorted(EXPECTED_PREPROCESSING_WARNING_IDS)}"
    )

if final_errors:
    print("预处理候选池完整性检查未通过：")

    for error in final_errors:
        print("  -", error)

    raise RuntimeError(
        "\n".join(
            final_errors
        )
    )

print("=" * 88)
print("预处理候选池完整性检查全部通过")
print("=" * 88)
print("历史预处理候选池497例全部成功生成NPZ并通过重新读取审计。")
print()
print("历史预处理元数据（仅用于候选池审计）：")
print("  A+B=269；C=228；不得作为最终分析队列定义。")
print()
print("下游最终分析必须从冻结的最终Master重新确定队列与标签：")
print("  Development = Centers B+C = 309")
print("  External = Center A = 180")
print("  Final analytic cohort = 489")